In [8]:
import pandas as pd

df = pd.read_csv("data_raw_fbref/dataset_con_kpis.csv")

# mínimo 10 partidos (900 min) para los rankings, más estricto que el filtro general
df_ranking = df[df["Playing Time_90s"] >= 10].copy()

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 10)

print("TOP 10 GOLEADORES POR 90 MINUTOS")
print(df_ranking.nlargest(10, "KPI_Goles_90")[
    ["Player", "Squad", "Comp", "Pos_Principal", "KPI_Goles_90", "Performance_Gls"]
].to_string(index=False))

print("\nTOP 10 CREADORES DE JUEGO POR xA/90")
print(df_ranking.nlargest(10, "KPI_xA_90")[
    ["Player", "Squad", "Comp", "Pos_Principal", "KPI_xA_90", "KPI_Asistencias_90"]
].to_string(index=False))

print("\nTOP 10 QUE MÁS SUPERAN LO ESPERADO (G-xG/90)")
print(df_ranking.nlargest(10, "KPI_G_menos_xG_90")[
    ["Player", "Squad", "Comp", "Pos_Principal", "KPI_G_menos_xG_90", "Performance_Gls"]
].to_string(index=False))

print("\nTOP 10 QUE MÁS RINDEN POR DEBAJO DE LO ESPERADO")
print(df_ranking.nsmallest(10, "KPI_G_menos_xG_90")[
    ["Player", "Squad", "Comp", "Pos_Principal", "KPI_G_menos_xG_90", "Performance_Gls"]
].to_string(index=False))

TOP 10 GOLEADORES POR 90 MINUTOS
            Player           Squad               Comp Pos_Principal  KPI_Goles_90  Performance_Gls
        Harry Kane   Bayern Munich      de Bundesliga            FW      1.363636             36.0
         Ansu Fati          Monaco         fr Ligue 1            MF      0.909091             11.0
 Jonathan Burkardt       Frankfurt      de Bundesliga            FW      0.878378             13.0
     Kylian Mbappé     Real Madrid         es La Liga            FW      0.865052             25.0
     Donyell Malen            Roma         it Serie A            FW      0.858896             14.0
          Raphinha       Barcelona         es La Liga            MF      0.849673             13.0
   Ousmane Dembélé             PSG         fr Ligue 1            FW      0.847458             10.0
    Erling Haaland Manchester City eng Premier League            FW      0.823171             27.0
       Deniz Undav       Stuttgart      de Bundesliga            FW      0.7

In [15]:
import pandas as pd

df = pd.read_csv("data_raw_fbref/dataset_final.csv")
df["Pos_Principal"] = df["Pos"].str.split(",").str[0]

columnas_numericas = [
    "Playing Time_90s", "Performance_Gls", "Performance_Ast", "Standard_G/Sh",
    "Ast", "Tackles_TklW", "Int", "xG", "xA", "npxG", "key_passes"
]

for col in columnas_numericas:
    if col in df.columns:
        df[col] = df[col].astype(str).str.replace(",", "").astype(float)

df["Playing Time_90s"] = df["Playing Time_90s"].replace(0, pd.NA)
n90 = df["Playing Time_90s"]

# KPIs Delanteros
df["KPI_Goles_90"] = df["Performance_Gls"] / n90
df["KPI_npxG_90"] = df["npxG"] / n90
df["KPI_G_menos_xG_90"] = (df["Performance_Gls"] - df["xG"]) / n90
df["KPI_Conversion_Tiros"] = df["Standard_G/Sh"]

# KPIs Mediocampistas
df["KPI_Asistencias_90"] = df["Performance_Ast"] / n90
df["KPI_xA_90"] = df["xA"] / n90
df["KPI_PasesClave_90"] = df["key_passes"] / n90

# KPIs Defensas (reconstruido: TklW + Int, ya que Tkl+Int original está vacío)
df["KPI_AccionesDefensivas_90"] = (df["Tackles_TklW"] + df["Int"]) / n90

columnas_kpi = [c for c in df.columns if c.startswith("KPI_")]
df.loc[df["Pos_Principal"] == "GK", columnas_kpi] = None

df.to_csv("data_raw_fbref/dataset_con_kpis.csv", index=False)
print(f"Total jugadores: {len(df)}")
print(f"Jugadores de campo con KPIs: {len(df[df['Pos_Principal'] != 'GK'])}")

Total jugadores: 1998
Jugadores de campo con KPIs: 1859


C:\Users\USUARIO\AppData\Local\Temp\ipykernel_7548\1736998917.py:30: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["KPI_AccionesDefensivas_90"] = (df["Tackles_TklW"] + df["Int"]) / n90


In [16]:
df = pd.read_csv("data_raw_fbref/dataset_con_kpis.csv")
for col in ["KPI_PasesClave_90", "KPI_AccionesDefensivas_90", "KPI_Goles_90", "KPI_Asistencias_90"]:
    print(f"{col}: {df[col].notna().sum()} valores no nulos de {len(df)}")

KPI_PasesClave_90: 1793 valores no nulos de 1998
KPI_AccionesDefensivas_90: 1859 valores no nulos de 1998
KPI_Goles_90: 1859 valores no nulos de 1998
KPI_Asistencias_90: 1859 valores no nulos de 1998


In [17]:
import pandas as pd

df = pd.read_csv("data_raw_fbref/dataset_con_kpis.csv")
df_campo = df[df["Pos_Principal"] != "GK"].copy()

pd.set_option("display.width", 120)

print("DISTRIBUCIÓN: Goles/90 por posición")
print(df_campo.groupby("Pos_Principal")["KPI_Goles_90"].describe()
      [["count", "mean", "std", "min", "25%", "50%", "75%", "max"]])

print("\nDISTRIBUCIÓN: Asistencias/90 por posición")
print(df_campo.groupby("Pos_Principal")["KPI_Asistencias_90"].describe()
      [["count", "mean", "std", "min", "25%", "50%", "75%", "max"]])

print("\nDISTRIBUCIÓN: Acciones defensivas/90 por posición")
print(df_campo.groupby("Pos_Principal")["KPI_AccionesDefensivas_90"].describe()
      [["count", "mean", "std", "min", "25%", "50%", "75%", "max"]])

print("\nDISTRIBUCIÓN: Pases clave/90 por posición")
print(df_campo.groupby("Pos_Principal")["KPI_PasesClave_90"].describe()
      [["count", "mean", "std", "min", "25%", "50%", "75%", "max"]])

DISTRIBUCIÓN: Goles/90 por posición
               count      mean       std  min       25%       50%       75%       max
Pos_Principal                                                                        
DF             637.0  0.041650  0.058178  0.0  0.000000  0.000000  0.068966  0.434783
FW             307.0  0.342373  0.205693  0.0  0.199203  0.312500  0.464346  1.363636
MF             915.0  0.130704  0.134069  0.0  0.000000  0.097087  0.196723  0.909091

DISTRIBUCIÓN: Asistencias/90 por posición
               count      mean       std  min       25%       50%       75%       max
Pos_Principal                                                                        
DF             637.0  0.049973  0.071046  0.0  0.000000  0.000000  0.081967  0.547945
FW             307.0  0.121182  0.114041  0.0  0.032950  0.101010  0.178571  0.659341
MF             915.0  0.118637  0.113136  0.0  0.031349  0.096774  0.179910  0.739300

DISTRIBUCIÓN: Acciones defensivas/90 por posición
          

In [18]:
import pandas as pd

df = pd.read_csv("data_raw_fbref/dataset_con_kpis.csv")
df_campo = df[df["Pos_Principal"] != "GK"].copy()

pd.set_option("display.width", 120)

print("PROMEDIO DE GOLES/90 POR LIGA (solo delanteros)")
delanteros = df_campo[df_campo["Pos_Principal"] == "FW"]
print(delanteros.groupby("Comp")["KPI_Goles_90"].agg(["mean", "count"]).sort_values("mean", ascending=False))

print("\nPROMEDIO DE PASES CLAVE/90 POR LIGA (solo mediocampistas)")
medios = df_campo[df_campo["Pos_Principal"] == "MF"]
print(medios.groupby("Comp")["KPI_PasesClave_90"].agg(["mean", "count"]).sort_values("mean", ascending=False))

print("\nEDAD PROMEDIO POR LIGA (todos los jugadores de campo)")
print(df_campo.groupby("Comp")["Age"].agg(["mean", "count"]).sort_values("mean"))

print("\nRENDIMIENTO POR RANGO DE EDAD (Goles/90 + Asistencias/90 combinados)")
df_campo["rango_edad"] = pd.cut(
    df_campo["Age"],
    bins=[0, 21, 24, 27, 30, 50],
    labels=["<=21", "22-24", "25-27", "28-30", "31+"]
)
df_campo["G+A_90"] = df_campo["KPI_Goles_90"] + df_campo["KPI_Asistencias_90"]
print(df_campo.groupby("rango_edad", observed=True)["G+A_90"].agg(["mean", "count"]))

PROMEDIO DE GOLES/90 POR LIGA (solo delanteros)
                        mean  count
Comp                               
es La Liga          0.395712     61
de Bundesliga       0.392342     51
fr Ligue 1          0.358034     57
eng Premier League  0.337305     56
it Serie A          0.264190     82

PROMEDIO DE PASES CLAVE/90 POR LIGA (solo mediocampistas)
                        mean  count
Comp                               
eng Premier League  1.275669    179
de Bundesliga       1.268569    160
es La Liga          1.242225    179
it Serie A          1.174705    199
fr Ligue 1          1.146740    167

EDAD PROMEDIO POR LIGA (todos los jugadores de campo)
                         mean  count
Comp                                
fr Ligue 1          24.884058    345
eng Premier League  25.251989    377
de Bundesliga       25.314371    334
it Serie A          25.763547    406
es La Liga          26.110831    397

RENDIMIENTO POR RANGO DE EDAD (Goles/90 + Asistencias/90 combinados)
     

In [19]:
import pandas as pd

df = pd.read_csv("data_raw_fbref/dataset_con_kpis.csv")
df_campo = df[df["Pos_Principal"] != "GK"].copy()
df_campo = df_campo[df_campo["Playing Time_90s"] >= 10].copy()

pd.set_option("display.width", 120)

kpi_por_posicion = {
    "FW": "KPI_Goles_90",
    "MF": "KPI_PasesClave_90",
    "DF": "KPI_AccionesDefensivas_90"
}

for pos, kpi in kpi_por_posicion.items():
    grupo = df_campo[df_campo["Pos_Principal"] == pos].copy()
    media = grupo[kpi].mean()
    std = grupo[kpi].std()
    grupo["z_score"] = (grupo[kpi] - media) / std

    print(f"=== OUTLIERS POSITIVOS EN {pos} (KPI: {kpi}) ===")
    print(grupo.nlargest(8, "z_score")[["Player", "Squad", "Comp", kpi, "z_score"]].to_string(index=False))

    print(f"\n=== OUTLIERS NEGATIVOS EN {pos} (KPI: {kpi}) ===")
    print(grupo.nsmallest(5, "z_score")[["Player", "Squad", "Comp", kpi, "z_score"]].to_string(index=False))
    print()

=== OUTLIERS POSITIVOS EN FW (KPI: KPI_Goles_90) ===
            Player           Squad               Comp  KPI_Goles_90  z_score
        Harry Kane   Bayern Munich      de Bundesliga      1.363636 5.067720
 Jonathan Burkardt       Frankfurt      de Bundesliga      0.878378 2.585966
     Kylian Mbappé     Real Madrid         es La Liga      0.865052 2.517810
     Donyell Malen            Roma         it Serie A      0.858896 2.486325
   Ousmane Dembélé             PSG         fr Ligue 1      0.847458 2.427828
    Erling Haaland Manchester City eng Premier League      0.823171 2.303617
       Deniz Undav       Stuttgart      de Bundesliga      0.763052 1.996153
Robert Lewandowski       Barcelona         es La Liga      0.756757 1.963956

=== OUTLIERS NEGATIVOS EN FW (KPI: KPI_Goles_90) ===
              Player         Squad               Comp  KPI_Goles_90   z_score
      Anthony Elanga     Newcastle eng Premier League      0.000000 -1.906324
       Álvaro Morata          Como         i

In [20]:
rudiger = df[df["Player"].str.contains("Rüdiger", case=False, na=False)]
print(rudiger[["Player", "Squad", "Playing Time_90s", "Tackles_TklW", "Int", "Performance_CrdY", "Performance_CrdR"]])

               Player        Squad  Playing Time_90s  Tackles_TklW  Int  Performance_CrdY  Performance_CrdR
1583  Antonio Rüdiger  Real Madrid              16.6           5.0  4.0                 0                 0


In [21]:
print(rudiger[["Player", "Squad", "Playing Time_MP", "Playing Time_Starts", "Playing Time_Min", "Performance_Gls"]])

# también revisamos si hay más de una fila para él (por transferencia u otro motivo)
todas_filas = df[df["Player"].str.contains("Rüdiger", case=False, na=False)]
print(f"\nCantidad de filas para Rüdiger: {len(todas_filas)}")

               Player        Squad  Playing Time_MP  Playing Time_Starts  Playing Time_Min  Performance_Gls
1583  Antonio Rüdiger  Real Madrid               18                   16            1490.0              1.0

Cantidad de filas para Rüdiger: 1
